# 922-Agent Shared-Policy PPO (Vasai) - density heatmap

In [ ]:
import os, gc, glob, pickle, time, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import joblib
import networkx as nx
from scipy.spatial import cKDTree
from collections import defaultdict
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
PROCESSED_GLOB = '../data generation/data/processed/batch_*.parquet'
STATIC_PATH    = '../data generation/data/processed/edges_static_scaled.parquet'
STATIC_RAW     = '../data generation/data/static/edges_static.parquet'
ADJ_DIR        = '../data generation/data/graph_adj'
SCALER_DIR     = '../data generation/data/scalers'
CKPT_PATH      = '../model training/models/best_model/best.pt'
DYN_COLS = ['current_speed','current_travel_time','confidence','incident','incident_type','incident_severity','incidents_nearby','hourly_rainfall_mm','monsoon_active','local_train_disruption','is_public_holiday','school_holiday','travel_time_ratio','congestion_level','delay_seconds','speed_ratio','time_of_day_sin','time_of_day_cos','day_of_week_sin','day_of_week_cos']
STA_COLS = ['road_type_enc','num_lanes','oneway','road_length','traffic_signal_count','signals_per_km']
TTR_IDX = DYN_COLS.index('travel_time_ratio')
CONG_IDX = DYN_COLS.index('congestion_level')
IN_FEATURES = len(DYN_COLS) + len(STA_COLS)
L = 24; H = 12; CHEB_K = 3
print(f'IN_FEATURES={IN_FEATURES} L={L} H={H}')

In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, k, dil):
        super().__init__()
        self.pad  = (k - 1) * dil
        self.conv = nn.Conv1d(in_ch, out_ch, k, dilation=dil, padding=self.pad)
    def forward(self, x):
        return self.conv(x)[:, :, :x.shape[-1]]

class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k, dil):
        super().__init__()
        self.c1  = CausalConv1d(in_ch,  out_ch, k, dil)
        self.c2  = CausalConv1d(out_ch, out_ch, k, dil)
        self.ln1 = nn.LayerNorm(out_ch)
        self.ln2 = nn.LayerNorm(out_ch)
        self.res = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.act = nn.GELU()
    def forward(self, x):                               # x: [N, C, L]
        r = self.res(x)
        h = self.act(self.ln1(self.c1(x).transpose(1,2)).transpose(1,2))
        h = self.ln2(self.c2(h).transpose(1,2)).transpose(1,2)
        return self.act(h + r)

def build_tcn(in_ch, channels, k, dilations):
    blocks, ch = [], in_ch
    for co, d in zip(channels, dilations):
        blocks.append(TCNBlock(ch, co, k, d)); ch = co
    return nn.Sequential(*blocks), ch   # returns (Sequential, out_channels)

class GWNLayer(nn.Module):
    """
    Graph WaveNet diffusion layer.
    Combines: identity + K-hop fwd + K-hop bwd + K-hop adaptive adjacency.
    mix_in dim = (3K+1) * in_dim
    """
    def __init__(self, in_dim, res_dim, skip_dim, K):
        super().__init__()
        self.K       = K
        mix_in       = (3 * K + 1) * in_dim
        self.filter  = nn.Linear(mix_in, res_dim)
        self.gate    = nn.Linear(mix_in, res_dim)
        self.res_fc  = nn.Linear(in_dim, res_dim)
        self.skip_fc = nn.Linear(res_dim, skip_dim)
        self.bn      = nn.BatchNorm1d(res_dim)

    def _khop(self, A, x):
        parts, h = [], x
        for _ in range(self.K):
            h = A @ h; parts.append(h)
        return torch.cat(parts, dim=-1)   # [N, K*D]

    def forward(self, x, Af, Ab, Aa):    # x,Af,Ab,Aa all [N,*]
        mix = torch.cat([x,
                         self._khop(Af, x),
                         self._khop(Ab, x),
                         self._khop(Aa, x)], dim=-1)
        h    = self.bn(torch.tanh(self.filter(mix)) * torch.sigmoid(self.gate(mix)))
        skip = self.skip_fc(h)
        return h + self.res_fc(x), skip

def get_adaptive_adj(emb1, emb2, global_idx):
    """Returns [N_present, N_present] soft adjacency from learned embeddings."""
    e1 = emb1(global_idx); e2 = emb2(global_idx)
    return F.softmax(F.relu(e1 @ e2.T), dim=-1)

class ChebConv(nn.Module):
    """
    Chebyshev spectral graph conv (STGCN).
    W shape: [K+1, in_dim, out_dim] — one weight matrix per polynomial.
    """
    def __init__(self, in_dim, out_dim, K):
        super().__init__()
        self.K  = K
        self.W  = nn.Parameter(torch.randn(K + 1, in_dim, out_dim) * 0.01)
        self.b  = nn.Parameter(torch.zeros(out_dim))
        self.ln = nn.LayerNorm(out_dim)

    def forward(self, x, cheb):
        """x: [N, in_dim]  cheb: list of K+1 tensors [N, N]"""
        N   = x.shape[0]
        out = x.new_zeros(N, self.W.shape[-1])
        for k, T_k in enumerate(cheb[:self.K + 1]):
            # T_k: [N_present, N_present], x: [N_present, D]
            out = out + (T_k @ x) @ self.W[k]
        return self.ln(F.relu(out + self.b))

class STGCNBlock(nn.Module):
    def __init__(self, in_dim, out_dim, K):
        super().__init__()
        self.conv = ChebConv(in_dim, out_dim, K)
        self.res  = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
    def forward(self, x, cheb):
        return self.conv(x, cheb) + self.res(x)

class RMSNorm(nn.Module):
    """Custom RMSNorm — works on all PyTorch versions. param key = 'w'."""
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.w   = nn.Parameter(torch.ones(d))
        self.eps = eps
    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return (x / rms) * self.w

class SelectiveSSM(nn.Module):
    """
    S6 core of Mamba: B, C, dt are all INPUT-DEPENDENT.
    Sequential scan over L steps — for L=24 this is fast even in pure Python.
    Params: x_proj, dt_proj, A_log, D
    """
    def __init__(self, d_inner, d_state=16, dt_min=0.001, dt_max=0.1):
        super().__init__()
        self.d_inner = d_inner
        self.d_state = d_state
        dt_rank      = math.ceil(d_inner / 16)
        self.dt_rank = dt_rank

        self.x_proj  = nn.Linear(d_inner, dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(dt_rank, d_inner, bias=True)

        # Init dt_proj bias so softplus(bias) ≈ uniform[dt_min, dt_max]
        dt_init = torch.exp(
            torch.rand(d_inner) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        )
        with torch.no_grad():
            self.dt_proj.bias.copy_(torch.log(torch.expm1(dt_init)))

        # A: HiPPO init, log-parameterized for numerical stability
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_inner, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D     = nn.Parameter(torch.ones(d_inner))

    def forward(self, x):
        """x: [N, L, d_inner]  →  y: [N, L, d_inner]"""
        N, L, D  = x.shape
        S        = self.d_state
        A        = -torch.exp(self.A_log.float())   # always negative → stable

        xBC_dt          = self.x_proj(x)            # [N, L, dt_rank+2S]
        dt_raw, Bs, Cs  = xBC_dt.split([self.dt_rank, S, S], dim=-1)
        dt              = F.softplus(self.dt_proj(dt_raw))   # [N, L, D]

        h  = x.new_zeros(N, D, S)
        ys = []
        for t in range(L):
            dt_t  = dt[:, t, :].unsqueeze(-1)                        # [N, D, 1]
            A_bar = torch.exp(dt_t * A.unsqueeze(0))                 # [N, D, S]
            B_bar = dt_t * Bs[:, t, :].unsqueeze(1)                  # [N, D, S]
            h     = A_bar * h + B_bar * x[:, t, :].unsqueeze(-1)    # [N, D, S]
            ys.append(torch.einsum('nds,ns->nd', h, Cs[:, t, :]))   # [N, D]
        y = torch.stack(ys, dim=1)                                   # [N, L, D]
        return y + x * self.D.unsqueeze(0).unsqueeze(0)

class MambaBlock(nn.Module):
    """Full Mamba block: RMSNorm → split → depthwise conv → SSM → gate → project → residual."""
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        d_inner       = int(expand * d_model)
        self.norm     = RMSNorm(d_model)
        self.in_proj  = nn.Linear(d_model, 2 * d_inner, bias=False)
        self.conv1d   = nn.Conv1d(d_inner, d_inner, d_conv,
                                  groups=d_inner, padding=d_conv - 1, bias=True)
        self.ssm      = SelectiveSSM(d_inner, d_state)
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)

    def forward(self, x):                               # [N, L, d_model]
        N, L, _ = x.shape
        residual = x
        x        = self.norm(x)
        xb, z    = self.in_proj(x).chunk(2, dim=-1)    # each [N, L, d_inner]
        xb       = F.silu(self.conv1d(xb.transpose(1, 2))[:, :, :L].transpose(1, 2))
        y        = self.ssm(xb) * F.silu(z)
        return self.out_proj(y) + residual

def make_out_head(skip_dim, H, n_out=2):
    return nn.Sequential(
        nn.ReLU(),
        nn.Linear(skip_dim, skip_dim // 2),
        nn.ReLU(),
        nn.Linear(skip_dim // 2, H * n_out),
    )

# ── TCN + GWN ─────────────────────────────────────────────────────────────────
class TCN_GWN(nn.Module):
    def __init__(self, n_global, in_features, eed=64,
                 tc=[128,256,256,256], tk=3, td=[1,2,4,8],
                 gd=256, gs=512, gl=4, gk=2, nc=10, H=12):
        super().__init__(); self.H=H
        self.edge_emb=nn.Embedding(n_global,eed); self.adap_emb1=nn.Embedding(n_global,nc); self.adap_emb2=nn.Embedding(n_global,nc)
        for e in [self.edge_emb,self.adap_emb1,self.adap_emb2]: nn.init.xavier_uniform_(e.weight)
        self.tcn,tcn_out=build_tcn(in_features+eed,tc,tk,td); self.tcn_to_gwn=nn.Linear(tcn_out,gd)
        self.gwn_list=nn.ModuleList([GWNLayer(gd,gd,gs,gk) for _ in range(gl)]); self.out_head=make_out_head(gs,H)
    def forward(self,x,Af,Ab,gidx):
        N,L,_=x.shape; emb=self.edge_emb(gidx).unsqueeze(1).expand(-1,L,-1); x=torch.cat([x,emb],dim=-1)
        h=self.tcn(x.permute(0,2,1))[:,:,-1]; h=F.gelu(self.tcn_to_gwn(h))
        Aa=get_adaptive_adj(self.adap_emb1,self.adap_emb2,gidx)
        sk=x.new_zeros(N,self.gwn_list[0].skip_fc.out_features)
        for l in self.gwn_list: h,s=l(h,Af,Ab,Aa); sk=sk+s
        return self.out_head(sk).view(N,self.H,-1)

# ── TCN + STGCN ───────────────────────────────────────────────────────────────
class TCN_STGCN(nn.Module):
    def __init__(self, n_global, in_features, eed=64,
                 tc=[128,256,256,256], tk=3, td=[1,2,4,8],
                 sd=256, sk=3, sl=4, H=12):
        super().__init__(); self.H=H
        self.edge_emb=nn.Embedding(n_global,eed); nn.init.xavier_uniform_(self.edge_emb.weight)
        self.tcn,tcn_out=build_tcn(in_features+eed,tc,tk,td); self.tcn_to_stgcn=nn.Linear(tcn_out,sd)
        self.spatial=nn.ModuleList([STGCNBlock(sd,sd,sk) for _ in range(sl)]); self.out_head=make_out_head(sd,H)
    def forward(self,x,cheb,gidx):
        N,L,_=x.shape; emb=self.edge_emb(gidx).unsqueeze(1).expand(-1,L,-1); x=torch.cat([x,emb],dim=-1)
        h=self.tcn(x.permute(0,2,1))[:,:,-1]; h=F.gelu(self.tcn_to_stgcn(h))
        for b in self.spatial: h=b(h,cheb)
        return self.out_head(h).view(N,self.H,-1)

# ── LSTM + GWN ────────────────────────────────────────────────────────────────
class LSTM_GWN(nn.Module):
    def __init__(self,n_global,in_features,eed=64,lh=512,ll=3,ld=0.15,gd=256,gs=512,gl=4,gk=2,nc=10,H=12):
        super().__init__(); self.H=H
        self.edge_emb=nn.Embedding(n_global,eed); self.adap_emb1=nn.Embedding(n_global,nc); self.adap_emb2=nn.Embedding(n_global,nc)
        for e in [self.edge_emb,self.adap_emb1,self.adap_emb2]: nn.init.xavier_uniform_(e.weight)
        self.lstm=nn.LSTM(in_features+eed,lh,ll,batch_first=True,dropout=ld if ll>1 else 0.0)
        self.lstm_to_gwn=nn.Linear(lh,gd); self.gwn_list=nn.ModuleList([GWNLayer(gd,gd,gs,gk) for _ in range(gl)]); self.out_head=make_out_head(gs,H)
    def forward(self,x,Af,Ab,gidx):
        N,L,_=x.shape; emb=self.edge_emb(gidx).unsqueeze(1).expand(-1,L,-1); x=torch.cat([x,emb],dim=-1)
        h=F.gelu(self.lstm_to_gwn(self.lstm(x)[0][:,-1,:]))
        Aa=get_adaptive_adj(self.adap_emb1,self.adap_emb2,gidx)
        sk=x.new_zeros(N,self.gwn_list[0].skip_fc.out_features)
        for l in self.gwn_list: h,s=l(h,Af,Ab,Aa); sk=sk+s
        return self.out_head(sk).view(N,self.H,-1)

# ── Mamba + GWN ───────────────────────────────────────────────────────────────
class Mamba_GWN(nn.Module):
    def __init__(self,n_global,in_features,eed=64,dm=256,ds=16,dc=4,ex=2,nl=4,gd=256,gs=512,gl=4,gk=2,nc=10,H=12):
        super().__init__(); self.H=H
        self.edge_emb=nn.Embedding(n_global,eed); self.adap_emb1=nn.Embedding(n_global,nc); self.adap_emb2=nn.Embedding(n_global,nc)
        for e in [self.edge_emb,self.adap_emb1,self.adap_emb2]: nn.init.xavier_uniform_(e.weight)
        self.input_proj=nn.Sequential(nn.Linear(in_features+eed,dm),nn.LayerNorm(dm),nn.GELU())
        self.mamba_blocks=nn.ModuleList([MambaBlock(dm,ds,dc,ex) for _ in range(nl)])
        self.final_norm=RMSNorm(dm); self.mamba_to_gwn=nn.Linear(dm,gd)
        self.gwn_list=nn.ModuleList([GWNLayer(gd,gd,gs,gk) for _ in range(gl)]); self.out_head=make_out_head(gs,H)
    def forward(self,x,Af,Ab,gidx):
        N,L,_=x.shape; emb=self.edge_emb(gidx).unsqueeze(1).expand(-1,L,-1); x=torch.cat([x,emb],dim=-1)
        h=self.input_proj(x)
        for blk in self.mamba_blocks: h=blk(h)
        h=F.gelu(self.mamba_to_gwn(self.final_norm(h)[:,-1,:]))
        Aa=get_adaptive_adj(self.adap_emb1,self.adap_emb2,gidx)
        sk=x.new_zeros(N,self.gwn_list[0].skip_fc.out_features)
        for l in self.gwn_list: h,s=l(h,Af,Ab,Aa); sk=sk+s
        return self.out_head(sk).view(N,self.H,-1)

print("All 4 model classes defined.")

In [ ]:
# ── Load adjacency / metadata ─────────────────────────────────────────────────
with open(f'{ADJ_DIR}/corridor_meta.pkl',       'rb') as f: corridor_meta       = pickle.load(f)
with open(f'{ADJ_DIR}/corridor_file_index.pkl', 'rb') as f: corridor_file_index = pickle.load(f)
with open(f'{ADJ_DIR}/edge_to_global_idx.pkl',  'rb') as f: edge_to_global_idx  = pickle.load(f)
N_GLOBAL_EDGES = len(edge_to_global_idx)
df_static      = pd.read_parquet(STATIC_PATH)
static_lookup  = df_static.set_index('edge_id')[STA_COLS].to_dict('index')
print(f'Global edges : {N_GLOBAL_EDGES:,}')
print(f'Corridors    : {len(corridor_meta)}')

def load_corridor_adj(cid):
    with open(corridor_file_index[cid], 'rb') as f:
        return pickle.load(f)

# ── WEEK / SPLIT BOUNDARIES (matching preprocess.ipynb exactly) ──────────────
WEEK_START     = pd.Timestamp('2024-07-01 00:00:00')
TRAIN_END_HOUR = 107   # Mon 00:00 → Fri 11:00
VAL_END_HOUR   = 143   # Fri 12:00 → Sat 23:00
# TEST = hours 144-167 (Sun)

def hour_idx(ts):
    return int((ts - WEEK_START).total_seconds() // 3600)

def get_corridor_tensor(batch_df, meta, split_label=None):
    """
    Load feature tensor for a corridor from one batch file.

    split_label=None  → load ALL 168 hours (used for evaluation so
                         build_windows has enough timesteps to make windows
                         whose predictions fall in the test period).
    split_label='test' etc → filter to that split only (used during training).

    WHY split_label=None FOR EVAL:
      Test split = Sunday = 24 hours per edge.
      build_windows needs T >= L+H = 36.  24 < 36 → zero windows.
      Loading all 168 hours gives T=168 → 132 windows per edge.
      We then keep only windows whose Y (prediction) falls on Sunday.

    Returns: data [N,T,F], present_eids, local_indices
    """
    edge_ids  = meta['edge_ids']
    local_map = meta['local_map']

    if split_label is None:
        mask = batch_df['edge_id'].isin(edge_ids)
    else:
        mask = batch_df['edge_id'].isin(edge_ids) & (batch_df['split'] == split_label)

    sub = batch_df[mask].copy()
    if len(sub) == 0:
        return None, None, None

    for col in STA_COLS:
        sub[col] = sub['edge_id'].map(
            lambda eid, c=col: static_lookup.get(eid, {}).get(c, 0.0)
        )

    sub['_order'] = sub['edge_id'].map(lambda eid: local_map.get(eid, 0))
    sub = sub.sort_values(['_order', 'timestamp']).reset_index(drop=True)

    present_eids  = sub['edge_id'].unique().tolist()
    local_indices = [local_map[eid] for eid in present_eids if eid in local_map]

    N_present = sub['_order'].nunique()
    T         = sub.groupby('edge_id').size().max()
    all_cols  = DYN_COLS + STA_COLS
    data      = sub[all_cols].values.reshape(N_present, T, len(all_cols)).astype(np.float32)

    return data, present_eids, local_indices


def build_windows(data, L, H):
    """[N,T,F] → X:[W,N,L,F], Y:[W,N,H,2]"""
    N, T, F_dim = data.shape
    n_w = T - L - H + 1
    if n_w <= 0:
        return None, None
    X = np.stack([data[:, t:t+L, :]                              for t in range(n_w)])
    Y = np.stack([data[:, t+L:t+L+H, :][:, :, [TTR_IDX, CONG_IDX]] for t in range(n_w)])
    return X, Y   # [W,N,L,F], [W,N,H,2]


def test_window_mask(T, L, H, split='test'):
    """
    Returns boolean array [n_windows] marking which sliding windows
    have their PREDICTION TARGET in the specified split.

    Window at position t:  X = hours[t : t+L],  Y = hours[t+L : t+L+H]
    For Y to be in 'test' (Sunday, hours 144-167):
        t + L >= 144   AND   t + L + H - 1 <= 167
        → t >= 120     AND   t <= 132

    For Y to be in 'val' (hours 108-143):
        t + L >= 108   AND   t + L + H - 1 <= 143
        → t >= 84      AND   t <= 108
    """
    n_w = T - L - H + 1
    if n_w <= 0:
        return np.zeros(0, dtype=bool)

    if split == 'test':
        y_start_min = VAL_END_HOUR + 1   # 144
        y_end_max   = 167
    elif split == 'val':
        y_start_min = TRAIN_END_HOUR + 1  # 108
        y_end_max   = VAL_END_HOUR         # 143
    else:
        raise ValueError(f"split must be 'test' or 'val', got {split!r}")

    mask = np.zeros(n_w, dtype=bool)
    for t in range(n_w):
        y_start = t + L          # hour index of first predicted step
        y_end   = t + L + H - 1  # hour index of last predicted step
        if y_start >= y_start_min and y_end <= y_end_max:
            mask[t] = True
    return mask


def get_adj_tensors(adj_c, local_indices, device):
    N_full = adj_c['N']
    idx    = np.array(local_indices, dtype=np.int64)
    if adj_c['A_fwd_dense'] is not None and len(idx) <= adj_c['A_fwd_dense'].shape[0]:
        A_fwd_full = adj_c['A_fwd_dense']
        A_bwd_full = adj_c['A_bwd_dense']
    else:
        import scipy.sparse as sp
        r,c,d = adj_c['A_fwd_coo']
        A_fwd_full = sp.coo_matrix((d,(r,c)),shape=(N_full,N_full)).toarray().astype(np.float32)
        r2,c2,d2 = adj_c['A_bwd_coo']
        A_bwd_full = sp.coo_matrix((d2,(r2,c2)),shape=(N_full,N_full)).toarray().astype(np.float32)
    return (torch.from_numpy(A_fwd_full[np.ix_(idx,idx)]).to(device),
            torch.from_numpy(A_bwd_full[np.ix_(idx,idx)]).to(device))


def get_cheb_tensors(adj_c, local_indices, device):
    idx  = np.array(local_indices, dtype=np.int64)
    cheb = adj_c['cheb']
    while len(cheb) < CHEB_K + 1:
        cheb = cheb + [np.eye(adj_c['N'], dtype=np.float32)]
    return [torch.from_numpy(T_k[np.ix_(idx,idx)]).to(device) for T_k in cheb[:CHEB_K+1]]


def global_idx_tensor(present_eids, device):
    return torch.tensor([edge_to_global_idx[eid] for eid in present_eids],
                        dtype=torch.long, device=device)

print('Data utilities + split-aware window masking ready.')
print(f'Test  windows per edge: t in [120..132] = 13 windows')
print(f'Val   windows per edge: t in [84..108]  = 25 windows')

In [ ]:
with open(f'{ADJ_DIR}/corridor_meta.pkl','rb') as f: corridor_meta = pickle.load(f)
with open(f'{ADJ_DIR}/corridor_file_index.pkl','rb') as f: corridor_file_index = pickle.load(f)
with open(f'{ADJ_DIR}/edge_to_global_idx.pkl','rb') as f: edge_to_global_idx = pickle.load(f)
N_GLOBAL_EDGES = len(edge_to_global_idx)
df_static = pd.read_parquet(STATIC_PATH)
static_lookup = df_static.set_index('edge_id')[STA_COLS].to_dict('index')
df_static_raw = pd.read_parquet(STATIC_RAW)
scaler_mm = joblib.load(f'{SCALER_DIR}/scaler_mm_dynamic.pkl')
def inv_ttr(v): return v*(scaler_mm.data_max_[0]-scaler_mm.data_min_[0])+scaler_mm.data_min_[0]
def inv_cong(v): return v*(scaler_mm.data_max_[1]-scaler_mm.data_min_[1])+scaler_mm.data_min_[1]
print(f'Global edges: {N_GLOBAL_EDGES:,}')

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
hp = ckpt['hparams']
def load_model_auto(ckpt, hp, ng, nf, dev, path):
    p = path.lower()
    if 'tcn_gwn' in p and 'stgcn' not in p:
        m = TCN_GWN(ng,nf,eed=hp.get('edge_emb_dim',64),tc=hp.get('tcn_channels',[128,256,256,256]),gd=hp.get('gwn_dim',256),gs=hp.get('gwn_skip',512),gl=hp.get('gwn_layers',4),gk=hp.get('gwn_K',2),nc=hp.get('node_emb_c',10),H=hp['H'])
    elif 'tcn_stgcn' in p:
        m = TCN_STGCN(ng,nf,eed=hp.get('edge_emb_dim',64),tc=hp.get('tcn_channels',[128,256,256,256]),sd=hp.get('stgcn_dim',256),sk=hp.get('stgcn_K',3),sl=hp.get('stgcn_layers',4),H=hp['H'])
    elif 'lstm_gwn' in p:
        m = LSTM_GWN(ng,nf,eed=hp.get('edge_emb_dim',64),lh=hp.get('lstm_hidden',512),ll=hp.get('lstm_layers',3),gd=hp.get('gwn_dim',256),gs=hp.get('gwn_skip',512),gl=hp.get('gwn_layers',4),gk=hp.get('gwn_K',2),nc=hp.get('node_emb_c',10),H=hp['H'])
    elif 'mamba_gwn' in p:
        m = Mamba_GWN(ng,nf,eed=hp.get('edge_emb_dim',64),dm=hp.get('d_model',256),ds=hp.get('d_state',16),dc=hp.get('d_conv',4),ex=hp.get('expand',2),nl=hp.get('n_mamba_layers',4),gd=hp.get('gwn_dim',256),gs=hp.get('gwn_skip',512),gl=hp.get('gwn_layers',4),gk=hp.get('gwn_K',2),nc=hp.get('node_emb_c',10),H=hp['H'])
    else: raise ValueError(f'Unknown: {path}')
    m.load_state_dict(ckpt['model_state']); return m.to(dev).eval()
stgnn_model = load_model_auto(ckpt, hp, N_GLOBAL_EDGES, IN_FEATURES, device, CKPT_PATH)
print(f'Model: {sum(p.numel() for p in stgnn_model.parameters()):,} params | epoch {ckpt["epoch"]}')

In [ ]:
VARIANTS = ['standard','no_signal','no_toll','no_signal_no_toll']
TOLL_TYPES = {'motorway','motorway_link','trunk','trunk_link'}
VARIANT_COLORS = {'standard':'#2196F3','no_signal':'#4CAF50','no_toll':'#FF9800','no_signal_no_toll':'#9C27B0'}
MAX_SPEED_MS = 100.0/3.6
BPR_A, BPR_B = 0.15, 4.0
CAP_PER_LANE = {'motorway':1800,'trunk':1500,'primary':1200,'secondary':900,
                'tertiary':700,'residential':400,'service':300,'unclassified':400}

def haversine_m(la1,lo1,la2,lo2):
    R=6_371_000; dl=math.radians(la2-la1); dn=math.radians(lo2-lo1)
    a=math.sin(dl/2)**2+math.cos(math.radians(la1))*math.cos(math.radians(la2))*math.sin(dn/2)**2
    return R*2*math.asin(math.sqrt(a))

def set_weights(graph, preds, variant='standard'):
    PEN=999999.0
    for u,v,d in graph.edges(data=True):
        eid=d.get('edge_id','')
        ttr=max(preds.get(eid,{}).get('pred_ttr',1.5),1.0)
        base=d['ff_tt']*ttr; pen=0
        if variant in ('no_signal','no_signal_no_toll') and d.get('has_signal'): pen+=PEN
        if variant in ('no_toll','no_signal_no_toll') and d.get('is_toll'): pen+=PEN
        d['weight']=base+pen; d['pred_ttr']=ttr

def astar_edges(graph, src, dst):
    def h(n,d):
        a,b=graph.nodes[n],graph.nodes[d]
        return haversine_m(a.get('lat',0),a.get('lon',0),b.get('lat',0),b.get('lon',0))/MAX_SPEED_MS
    try: path=nx.astar_path(graph,src,dst,heuristic=h,weight='weight')
    except: return None
    return [(path[i],path[i+1]) for i in range(len(path)-1)]

def route_time(G,e): return sum(G[u][v].get('weight',G[u][v]['ff_tt']) for u,v in e) if e else 0.0
def route_dist(G,e): return sum(G[u][v].get('road_len',50) for u,v in e) if e else 0.0
def route_polyline(G,e):
    if not e: return []
    p=[(G.nodes[e[0][0]].get('lat',0),G.nodes[e[0][0]].get('lon',0))]
    for u,v in e: p.append((G.nodes[v].get('lat',0),G.nodes[v].get('lon',0)))
    return p

def candidate_routes(graph, src, dst, K=5, rng=None):
    if rng is None: rng=np.random.default_rng()
    cs=[]; seen=set()
    r0=astar_edges(graph,src,dst)
    if r0 is None: return cs
    cs.append(r0); seen.add(tuple(r0))
    ow={(u,v):d['weight'] for u,v,d in graph.edges(data=True)}
    for _ in range(K*4):
        if len(cs)>=K: break
        for u,v,d in graph.edges(data=True): d['weight']=ow[(u,v)]*rng.uniform(0.7,1.5)
        r=astar_edges(graph,src,dst)
        if r and tuple(r) not in seen: cs.append(r); seen.add(tuple(r))
    for u,v,d in graph.edges(data=True): d['weight']=ow[(u,v)]
    return cs

def compute_loads(routes):
    ld=defaultdict(int)
    for r in routes:
        for u,v in r: ld[(u,v)]+=1
    return ld

def bpr_time(G, edges, loads):
    t=0.0
    for u,v in edges:
        d=G[u][v]; vol=loads.get((u,v),0); cap=max(d.get('capacity',400),1)
        t+=d['ff_tt']*(1.0+BPR_A*(vol/cap)**BPR_B)
    return t

def impose_congestion(graph, all_route0s, rng, frac=0.6, mult_range=(1.5, 3.0)):
    """
    IMPOSE congestion on A* route-0 edges.
    Simulates: everyone follows A* -> those roads get congested.
    60% of route-0 edges get volume = capacity * random(1.5, 3.0).
    This makes BPR actually bite: time multiplier = 1 + 0.15 * (1.5-3.0)^4 = 1.76 to 12.3x
    """
    route0_edges = set()
    for r in all_route0s:
        for u,v in r: route0_edges.add((u,v))
    route0_list = list(route0_edges)
    n_congest = max(1, int(len(route0_list) * frac))
    chosen_idx = rng.choice(len(route0_list), size=min(n_congest, len(route0_list)), replace=False)
    loads = {}
    for idx in chosen_idx:
        u, v = route0_list[idx]
        cap = max(graph[u][v].get('capacity', 400), 1)
        loads[(u,v)] = int(cap * rng.uniform(mult_range[0], mult_range[1]))
    return loads

def realized_cong_map(G, loads):
    rc={}
    for u,v,d in G.edges(data=True):
        vol=loads.get((u,v),0); cap=max(d.get('capacity',400),1)
        bpr=1.0+BPR_A*(vol/cap)**BPR_B
        rc[(u,v)]=1.0-1.0/bpr
    return rc

def build_subgraph(df_raw, lat_range, lon_range):
    mask=((df_raw['lat']>=lat_range[0])&(df_raw['lat']<=lat_range[1])&
          (df_raw['lon']>=lon_range[0])&(df_raw['lon']<=lon_range[1]))
    df=df_raw[mask].copy().reset_index(drop=True)
    eids=set(df['edge_id'].tolist())
    G=nx.DiGraph(); nl,no={},{}
    for _,row in df.iterrows():
        u,v=int(row['u']),int(row['v'])
        ffs=float(row['free_flow_speed']); rl=float(row['road_length'])
        ft=max(rl/(ffs/3.6),0.1); rt=str(row.get('road_type','unclassified'))
        sg=int(row.get('traffic_signal_count',0)); lanes=max(int(row.get('num_lanes',1)),1)
        la,lo=float(row['lat']),float(row['lon']); rb=rt.split('_')[0]
        G.add_edge(u,v,edge_id=row['edge_id'],ff_tt=ft,road_len=rl,ffs_kmh=ffs,
                   road_type=rt,signals=sg,has_signal=sg>0,is_toll=rt in TOLL_TYPES,
                   num_lanes=lanes,lat=la,lon=lo,capacity=CAP_PER_LANE.get(rb,400)*lanes)
        for n in [u,v]:
            nl.setdefault(n,[]).append(la); no.setdefault(n,[]).append(lo)
    for n in G.nodes():
        G.nodes[n]['lat']=np.mean(nl.get(n,[0])); G.nodes[n]['lon']=np.mean(no.get(n,[0]))
    nids=[]; nco=[]
    for n,d in G.nodes(data=True):
        if 'lat' in d: nids.append(n); nco.append([d['lat'],d['lon']])
    nids=np.array(nids); nco=np.array(nco); kdt=cKDTree(nco)
    return G,eids,nids,nco,kdt,df

def infer_subgraph(model, dep_hour, target_eids, device, ckpt_path):
    WS=pd.Timestamp('2024-07-01')
    ts0=WS+pd.Timedelta(hours=dep_hour-L); ts1=WS+pd.Timedelta(hours=dep_hour-1)
    is_stg='stgcn' in ckpt_path.lower(); preds={}
    for bp in sorted(glob.glob(PROCESSED_GLOB)):
        bdf=pd.read_parquet(bp); bdf['timestamp']=pd.to_datetime(bdf['timestamp'])
        bw=bdf[(bdf['timestamp']>=ts0)&(bdf['timestamp']<=ts1)].copy()
        if len(bw)==0: del bdf; continue
        for cid,meta in corridor_meta.items():
            if not target_eids.intersection(meta['edge_ids']): continue
            em=meta['local_map']; sub=bw[bw['edge_id'].isin(meta['edge_ids'])].copy()
            if len(sub)==0: continue
            for c in STA_COLS: sub[c]=sub['edge_id'].map(lambda e,c=c:static_lookup.get(e,{}).get(c,0.0))
            sub['_o']=sub['edge_id'].map(lambda e:em.get(e,0))
            sub=sub.sort_values(['_o','timestamp']).reset_index(drop=True)
            eids=sub['edge_id'].unique().tolist()
            lidx=[em[e] for e in eids if e in em]
            Np=sub['_o'].nunique(); T=sub.groupby('edge_id').size().max()
            if T<L or Np<2: continue
            data=sub[DYN_COLS+STA_COLS].values.reshape(Np,T,IN_FEATURES).astype(np.float32)[:,-L:,:]
            ac=load_corridor_adj(cid); gi=global_idx_tensor(eids,device)
            with torch.no_grad():
                xt=torch.from_numpy(data).to(device)
                if is_stg:
                    cheb=get_cheb_tensors(ac,lidx,device); pred=model(xt,cheb,gi)
                else:
                    Af,Ab=get_adj_tensors(ac,lidx,device); pred=model(xt,Af,Ab,gi)
            pn=pred.cpu().numpy()
            for i,eid in enumerate(eids):
                if eid in target_eids:
                    preds[eid]={'pred_ttr':float(inv_ttr(pn[i,0,0])),'pred_cong':float(inv_cong(pn[i,0,1]))}
            del ac; torch.cuda.empty_cache()
        del bdf,bw; gc.collect()
    return preds

class PPOPolicy(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.shared=nn.Sequential(nn.Linear(state_dim,hidden),nn.ReLU(),nn.Linear(hidden,hidden),nn.ReLU())
        self.actor=nn.Sequential(nn.Linear(hidden,hidden//2),nn.ReLU(),nn.Linear(hidden//2,action_dim))
        self.critic=nn.Sequential(nn.Linear(hidden,hidden//2),nn.ReLU(),nn.Linear(hidden//2,1))
    def forward(self,s):
        if s.dim()==1: s=s.unsqueeze(0)
        f=self.shared(s); return self.actor(f),self.critic(f)
    def act(self,s,det=False):
        lo,val=self.forward(s); dist=torch.distributions.Categorical(logits=lo)
        a=lo.argmax(-1) if det else dist.sample()
        return a.item(),dist.log_prob(a).squeeze(),val.squeeze()
    def evaluate(self,states,actions):
        lo,v=self.forward(states); dist=torch.distributions.Categorical(logits=lo)
        return dist.log_prob(actions),v.squeeze(-1),dist.entropy()

def ppo_update(policy,opt,buf,device,clip=0.2,inner=4,batch=64,ent_c=0.01,val_c=0.5,gc_norm=0.5):
    S=torch.from_numpy(buf['states']).float().to(device)
    A=torch.from_numpy(buf['actions']).long().to(device)
    R=torch.from_numpy(buf['rewards']).float().to(device)
    OLP=torch.from_numpy(buf['log_probs']).float().to(device)
    adv=R-torch.from_numpy(buf['values']).float().to(device)
    adv=(adv-adv.mean())/(adv.std()+1e-8)
    N=len(S); tpl,tvl,te,nu=0,0,0,0
    policy.train()
    for _ in range(inner):
        idx=torch.randperm(N)
        for s in range(0,N,batch):
            b=idx[s:s+batch]
            nlp,nv,ent=policy.evaluate(S[b],A[b])
            ratio=torch.exp(nlp-OLP[b])
            s1=ratio*adv[b]; s2=torch.clamp(ratio,1-clip,1+clip)*adv[b]
            pl=-torch.min(s1,s2).mean(); vl=F.mse_loss(nv,R[b])
            loss=pl+val_c*vl-ent_c*ent.mean()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(),gc_norm); opt.step()
            tpl+=pl.item(); tvl+=vl.item(); te+=ent.mean().item(); nu+=1
    return {'ploss':tpl/nu,'vloss':tvl/nu,'entropy':te/nu}

print('A*, BPR, impose_congestion, PPO all defined.')

In [ ]:
# Cache predictions to disk
CACHE_FILE = 'vasai_pred_cache.pkl'
G, sub_eids, nids, nco, kd, df_sub = build_subgraph(df_static_raw, (19.33,19.42), (72.79,72.86))
print(f'Vasai: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
VALID_HOURS = list(range(L, 168))
if os.path.exists(CACHE_FILE):
    print(f'Loading cache from {CACHE_FILE}...')
    with open(CACHE_FILE, 'rb') as f: pcache = pickle.load(f)
    print(f'Loaded {len(pcache)} timestamps')
else:
    print(f'Computing {len(VALID_HOURS)} timestamps (saves to {CACHE_FILE})...')
    pcache = {}; t0 = time.perf_counter()
    for hi, hour in enumerate(VALID_HOURS):
        pcache[hour] = infer_subgraph(stgnn_model, hour, sub_eids, device, CKPT_PATH)
        if (hi+1)%24==0: print(f'  [{hi+1}/{len(VALID_HOURS)}] {time.perf_counter()-t0:.0f}s')
    with open(CACHE_FILE, 'wb') as f: pickle.dump(pcache, f)
    print(f'Saved to {CACHE_FILE}')

In [ ]:
CKPT_DIR='models/ppo_922'; EP_DIR='episode_maps_922'
os.makedirs(CKPT_DIR,exist_ok=True); os.makedirs(EP_DIR,exist_ok=True)
N_AGENTS=922; K=5; MIN_DIST=400; D_ROUTE=10; D_GLOBAL=5
STATE_DIM=K*D_ROUTE+D_GLOBAL; ACTION_DIM=K
ACOLS=['#E53935','#1E88E5','#43A047','#FB8C00','#8E24AA','#00ACC1','#C0CA33','#6D4C41','#546E7A','#D81B60']

class MultiEnv:
    def __init__(self,graph,pcache,hours,node_ids,kd_t,seed=42):
        self.G=graph; self.pc=pcache; self.hrs=hours; self.nids=node_ids; self.kd=kd_t
        self.rng=np.random.default_rng(seed); self.nn=len(node_ids); self.ep=0
    def _pair(self):
        for _ in range(100):
            i,j=self.rng.choice(self.nn,2,replace=False); s,d=int(self.nids[i]),int(self.nids[j])
            if haversine_m(self.G.nodes[s].get('lat',0),self.G.nodes[s].get('lon',0),
                           self.G.nodes[d].get('lat',0),self.G.nodes[d].get('lon',0))>=MIN_DIST: return s,d
        return int(self.nids[0]),int(self.nids[1])
    def reset(self,hour_idx=None):
        self.ep+=1
        if hour_idx is None: hour_idx=self.rng.choice(self.hrs)
        self.hour=hour_idx; self.preds=self.pc.get(hour_idx,{})
        self.agents=[]
        for i in range(N_AGENTS):
            s,d=self._pair(); v=VARIANTS[i%len(VARIANTS)]
            set_weights(self.G,self.preds,variant=v)
            cs=candidate_routes(self.G,s,d,K=K,rng=self.rng)
            if not cs: s,d=self._pair(); cs=candidate_routes(self.G,s,d,K=K,rng=self.rng)
            while len(cs)<K: cs.append(cs[-1] if cs else [])
            self.agents.append({'src':s,'dst':d,'variant':v,'cands':cs[:K]})
        all_r0=[ag['cands'][0] for ag in self.agents if ag['cands'][0]]
        self.loads=impose_congestion(self.G,all_r0,self.rng,frac=0.6,mult_range=(1.5,3.0))
        self.rc=realized_cong_map(self.G,self.loads)
        return self._states()
    def _agent_state(self,i):
        ag=self.agents[i]; rf=np.zeros((K,D_ROUTE),dtype=np.float32)
        for k,route in enumerate(ag['cands']):
            if not route: continue
            pt=route_time(self.G,route); dist=route_dist(self.G,route); ns=len(route)
            pc=[max(self.preds.get(self.G[u][v].get('edge_id',''),{}).get('pred_cong',0.3),0) for u,v in route]
            sigs=sum(self.G[u][v].get('signals',0) for u,v in route)
            rl=sum(self.loads.get((u,v),0) for u,v in route)
            bm=np.mean([1+BPR_A*(self.loads.get((u,v),0)/max(self.G[u][v].get('capacity',400),1))**BPR_B for u,v in route])
            rcv=[self.rc.get((u,v),0.0) for u,v in route]
            rf[k]=[pt/600,dist/5000,ns/50,np.mean(pc),np.max(pc),sigs/10,rl/max(ns,1)/2000,bm/2,np.mean(rcv),np.max(rcv)]
        h=self.hour%24; dow=(self.hour//24)%7
        gf=np.array([np.sin(2*np.pi*h/24),np.cos(2*np.pi*h/24),np.sin(2*np.pi*dow/7),np.cos(2*np.pi*dow/7),1.0],dtype=np.float32)
        return np.concatenate([rf.flatten(),gf])
    def _states(self): return [self._agent_state(i) for i in range(N_AGENTS)]
    def step_round1(self,actions):
        routes=[self.agents[i]['cands'][min(int(a),K-1)] for i,a in enumerate(actions)]
        for i,r in enumerate(routes): self.agents[i]['r1']=r
        self.loads=impose_congestion(self.G,routes,self.rng,frac=0.5,mult_range=(1.2,2.5))
        self.rc=realized_cong_map(self.G,self.loads); return self._states()
    def step_round2(self,actions):
        routes=[self.agents[i]['cands'][min(int(a),K-1)] for i,a in enumerate(actions)]
        fl=impose_congestion(self.G,routes,self.rng,frac=0.5,mult_range=(1.2,2.5)); results=[]
        for i in range(N_AGENTS):
            ch=routes[i]; bl=self.agents[i]['cands'][0]
            at=bpr_time(self.G,ch,fl); bt=bpr_time(self.G,bl,fl); pred_tt=route_time(self.G,bl)
            imp=(bt-at)/max(bt,1); rew=-(at/600)+2*imp
            cd=route_dist(self.G,ch); bd=route_dist(self.G,bl)
            if cd>2*bd: rew-=0.3
            results.append({'reward':rew,'actual_tt':at,'baseline_tt':bt,'pred_tt':pred_tt,
                           'improvement':imp,'action':int(actions[i]),'chosen':ch,'baseline':bl,
                           'src':self.agents[i]['src'],'dst':self.agents[i]['dst'],'variant':self.agents[i]['variant']})
        return results

def save_ep_pngs(G,results,ep_num,hour,save_dir):
    fig,ax=plt.subplots(figsize=(12,12))
    for u,ve,d in G.edges(data=True):
        n1,n2=G.nodes[u],G.nodes[ve]; ax.plot([n1['lon'],n2['lon']],[n1['lat'],n2['lat']],color='#eee',lw=0.3)
    eu=defaultdict(int)
    for res in results:
        for u,v in res['chosen']: eu[(u,v)]+=1
    mx=max(eu.values()) if eu else 1
    for (u,v),cnt in eu.items():
        n1,n2=G.nodes[u],G.nodes[v]; f=cnt/mx
        ax.plot([n1['lon'],n2['lon']],[n1['lat'],n2['lat']],color=plt.cm.YlOrRd(0.2+0.8*f),lw=0.5+3.5*f,alpha=0.85)
    aimp=np.mean([r['improvement'] for r in results]); wins=sum(1 for r in results if r['improvement']>0)
    ax.set_title(f'Ep {ep_num} | 922 agents | {aimp:+.1%} | Wins {wins}/922',fontweight='bold')
    ax.legend(handles=[mpatches.Patch(color=plt.cm.YlOrRd(0.3),label='Low'),mpatches.Patch(color=plt.cm.YlOrRd(0.9),label=f'High(max={mx})')],loc='upper left')
    plt.tight_layout(); plt.savefig(f'{save_dir}/ep_{ep_num:04d}.png',dpi=100,bbox_inches='tight'); plt.close()
    return  # density heatmap only, no 2-panel
env=MultiEnv(G,pcache,VALID_HOURS,nids,nco,seed=42)
policy=PPOPolicy(STATE_DIM,ACTION_DIM,hidden=128).to(device)
opt=optim.Adam(policy.parameters(),lr=3e-4)
print(f'Env: {N_AGENTS} agents, state={STATE_DIM}, action={ACTION_DIM}')

EP_PER=100; EPOCHS=50; EVAL_N=50; SAVE_EVERY=50
tr_r,tr_imp,ev_r,ev_imp,losses=[],[],[],[],[]; best=-float('inf'); gep=0
for epoch in range(1,EPOCHS+1):
    t0=time.perf_counter(); all_s,all_a,all_r,all_lp,all_v,all_info=[],[],[],[],[],[]
    policy.eval()
    for i in range(EP_PER):
        ep=gep+i; s1=env.reset()
        a1=[policy.act(torch.from_numpy(s1[j]).float().to(device))[0] for j in range(N_AGENTS)]
        s2=env.step_round1(a1); a2=[]; lps=[]; vals=[]
        for j in range(N_AGENTS):
            st=torch.from_numpy(s2[j]).float().to(device); a,lp,v=policy.act(st)
            a2.append(a); lps.append(lp.item()); vals.append(v.item())
        res=env.step_round2(a2)
        for j in range(N_AGENTS):
            all_s.append(s2[j]); all_a.append(a2[j]); all_r.append(res[j]['reward'])
            all_lp.append(lps[j]); all_v.append(vals[j])
        all_info.append(res)
        if ep%SAVE_EVERY==0: save_ep_pngs(env.G,res,ep,env.hour,EP_DIR)
    gep+=EP_PER
    buf={'states':np.array(all_s,np.float32),'actions':np.array(all_a,np.int64),
         'rewards':np.array(all_r,np.float32),'log_probs':np.array(all_lp,np.float32),
         'values':np.array(all_v,np.float32)}
    tr_r.append(buf['rewards'].mean()); tr_imp.append(np.mean([r['improvement'] for er in all_info for r in er]))
    ls=ppo_update(policy,opt,buf,device); losses.append(ls)
    policy.eval(); er,ei=[],[]
    for _ in range(EVAL_N):
        s1=env.reset()
        a1=[policy.act(torch.from_numpy(s1[j]).float().to(device),det=True)[0] for j in range(N_AGENTS)]
        s2=env.step_round1(a1)
        a2=[policy.act(torch.from_numpy(s2[j]).float().to(device),det=True)[0] for j in range(N_AGENTS)]
        res=env.step_round2(a2)
        for r in res: er.append(r['reward']); ei.append(r['improvement'])
    ev_r.append(np.mean(er)); ev_imp.append(np.mean(ei))
    print(f'E{epoch:3d}/{EPOCHS} tr={tr_r[-1]:+.3f} ev={ev_r[-1]:+.3f} imp={ev_imp[-1]:+.1%} {time.perf_counter()-t0:.0f}s')
    if ev_r[-1]>best:
        best=ev_r[-1]; torch.save({'epoch':epoch,'model_state':policy.state_dict(),'eval_reward':best,
                                    'state_dim':STATE_DIM,'action_dim':ACTION_DIM},f'{CKPT_DIR}/best.pt'); print('  saved')
matplotlib.use('TkAgg')
fig,axes=plt.subplots(2,2,figsize=(14,10)); er=range(1,EPOCHS+1)
axes[0,0].plot(er,tr_r,lw=2,label='Train'); axes[0,0].plot(er,ev_r,lw=2,label='Eval')
axes[0,0].axhline(0,color='gray',ls='--'); axes[0,0].set_title('Reward'); axes[0,0].legend()
axes[0,1].plot(er,[x*100 for x in tr_imp],lw=2,label='Train'); axes[0,1].plot(er,[x*100 for x in ev_imp],lw=2,label='Eval')
axes[0,1].axhline(0,color='gray',ls='--'); axes[0,1].set_title('Improvement (%)'); axes[0,1].legend()
axes[1,0].plot(er,[l['ploss'] for l in losses],lw=2,color='mediumpurple'); axes[1,0].set_title('Policy Loss')
axes[1,1].plot(er,[l['entropy'] for l in losses],lw=2,color='seagreen'); axes[1,1].set_title('Entropy')
plt.suptitle('922-Agent Shared PPO (Vasai)',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig(f'{CKPT_DIR}/curves.png',dpi=120); plt.show()
print(f'Done. Best={best:+.3f}. PNGs: {EP_DIR}/')